# 02 - Modeling and Evaluation

## Objective

This notebook trains and compares several models for Probability of Default prediction:

- Logistic Regression
- Custom GLMBoost-inspired model
- Tree-based Gradient Boosting model

The goal is to compare predictive performance and interpretability.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path(r"C:\Users\Akkar\OneDrive\Bureau\credit-risk-pd-modeling-clean")
sys.path.append(str(PROJECT_ROOT))

from src.glmboost_model import SimpleGLMBoost

In [ ]:
DATA_PATH = "../data/echantillon_etu_tr_2.csv"

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df_model = df.copy()

if "ID" in df_model.columns:
    df_model = df_model.drop(columns=["ID"])

if "tx_imp" in df_model.columns:
    df_model = df_model.drop(columns=["tx_imp"])

if "ancienneté1" in df_model.columns:
    df_model["ancienneté1"] = df_model["ancienneté1"].fillna(df_model["ancienneté1"].mean())

if "engagement2" in df_model.columns:
    df_model["engagement2"] = df_model["engagement2"].fillna(df_model["engagement2"].mean())

if "type_imp" in df_model.columns:
    df_model["type_imp"] = df_model["type_imp"].fillna("No incident")

In [ ]:
X = df_model.drop("défaut", axis=1)
y = df_model["défaut"]

X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
def evaluate_model(model, X_test_data, y_test_data, model_name):
    y_pred = model.predict(X_test_data)
    y_prob = model.predict_proba(X_test_data)[:, 1]

    print(f"Model: {model_name}")
    print("Accuracy:", accuracy_score(y_test_data, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test_data, y_prob))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test_data, y_pred))
    print("\nClassification Report:\n", classification_report(y_test_data, y_pred))

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test_data, y_pred),
        "ROC-AUC": roc_auc_score(y_test_data, y_prob),
    }

In [ ]:
logistic_model = LogisticRegression(max_iter=5000)

logistic_model.fit(X_train_scaled, y_train)

logistic_results = evaluate_model(
    logistic_model,
    X_test_scaled,
    y_test,
    "Logistic Regression"
)

In [ ]:
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_results = evaluate_model(
    gb_model,
    X_test,
    y_test,
    "Tree-Based Gradient Boosting"
)

In [ ]:
custom_glmboost = SimpleGLMBoost(
    n_estimators=1000,
    learning_rate=0.1
)

custom_glmboost.fit(X_train_scaled, y_train.values)

glmboost_results = evaluate_model(
    custom_glmboost,
    X_test_scaled,
    y_test,
    "Custom GLMBoost-Inspired Model"
)

In [ ]:
comparison = pd.DataFrame([
    logistic_results,
    glmboost_results,
    gb_results
])

comparison

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": gb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.gca().invert_yaxis()
plt.title("Feature Importance - Tree-Based Gradient Boosting")
plt.xlabel("Importance")
plt.grid(True)
plt.show()

## Modeling conclusion

The tree-based Gradient Boosting model generally provides the best predictive performance, while Logistic Regression remains the most interpretable model.

The custom GLMBoost-inspired model provides an intermediate approach between interpretability and predictive power.